cell_typemetadatasource

In [ ]:
from pathlib import Path
import os

# Works whether Jupyter starts in the repository root or this notebook's folder.
_candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in _candidates if (p / "data" / "raw").exists() and (p / "Machine Learning Pipeline").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the repository root. Open this notebook from inside the T006 project folder.")
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)

## 1. Load raw files and inspect them

In [ ]:
import json
import re
import numpy as np
import pandas as pd
from IPython.display import display

metadata_path = DATA_RAW / "metadata_ideal.json"
metadata_csv_path = DATA_RAW / "metadata.csv"
flow_path = DATA_RAW / "UrbanTALES-flowdata.csv"

for path in [metadata_path, metadata_csv_path, flow_path]:
    if not path.exists():
        raise FileNotFoundError(f"Required raw data file is missing: {path}. Copy the provided raw UrbanTALES files into data/raw/ and rerun this notebook.")

with metadata_path.open(encoding="utf-8") as f:
    ideal_metadata = json.load(f)
metadata_catalogue = pd.read_csv(metadata_csv_path)
flowdata = pd.read_csv(flow_path)

print("Idealized metadata cases:", len(ideal_metadata))
print("Full metadata table:", metadata_catalogue.shape)
print("Full flow-data table:", flowdata.shape)
print("Flow-data columns:", flowdata.columns.tolist())
display(metadata_catalogue.head(3))
display(flowdata.head(3))

## 2. Inspect the three raw ZIP archives (profiles, topology and NetCDF fields)

In [ ]:
import zipfile
from io import BytesIO
import h5py

archive_specs = {
    "profiles": DATA_RAW / "selected_files.zip",
    "topology": DATA_RAW / "selected_files (1).zip",
    "netcdf": DATA_RAW / "selected_files (2).zip",
}
for archive_name, archive_path in archive_specs.items():
    if not archive_path.exists():
        raise FileNotFoundError(f"Required supplied raw archive is missing: {archive_path}")

archive_rows = []
profile_crosscheck_rows = []
topology_rows = []
netcdf_rows = []

# Read every member of each ZIP. Keep the original ZIPs intact; no manual extraction is needed.
with zipfile.ZipFile(archive_specs["profiles"]) as zf:
    profile_members = [name for name in zf.namelist() if name.lower().endswith(".csv") and not name.endswith("/")]
    for member in profile_members:
        case_id = member.split("/")[0]
        profile = pd.read_csv(BytesIO(zf.read(member)))
        archive_rows.append({"archive": "profiles", "member": member, "case_id": case_id, "bytes": zf.getinfo(member).file_size})
        if {"z", "U"}.issubset(profile.columns):
            profile_numeric = profile[["z", "U"]].apply(pd.to_numeric, errors="coerce").dropna().sort_values("z")
            meta_match = next((item for item in ideal_metadata if item["Folder Name"] == case_id), None)
            if meta_match is not None and len(profile_numeric) >= 2:
                h_mean = float(np.mean(meta_match["Heights"]))
                z_values = profile_numeric["z"].to_numpy(dtype=float)
                u_values = profile_numeric["U"].to_numpy(dtype=float)
                inside = (z_values > 0) & (z_values < h_mean)
                z_integral = np.concatenate(([0.0], z_values[inside], [h_mean]))
                u_integral = np.concatenate(([np.interp(0.0, z_values, u_values)], u_values[inside], [np.interp(h_mean, z_values, u_values)]))
                profile_mean_u = float(np.trapz(u_integral, z_integral) / h_mean)
                profile_crosscheck_rows.append({
                    "simulation_id": case_id,
                    "profile_member": member,
                    "profile_integrated_U_m_s": profile_mean_u,
                    "flow_table_U_m_s": float(flowdata.loc[flowdata["NameE"].astype(str) == case_id, "U"].iloc[0]) if (flowdata["NameE"].astype(str) == case_id).any() else np.nan,
                    "mean_building_height_m": h_mean,
                    "absolute_difference_m_s": abs(profile_mean_u - float(flowdata.loc[flowdata["NameE"].astype(str) == case_id, "U"].iloc[0])) if (flowdata["NameE"].astype(str) == case_id).any() else np.nan,
                })

with zipfile.ZipFile(archive_specs["topology"]) as zf:
    topology_members = [name for name in zf.namelist() if not name.endswith("/")]
    for member in topology_members:
        raw = zf.read(member)
        try:
            grid = np.loadtxt(BytesIO(raw), dtype=float)
            grid_shape = tuple(int(x) for x in grid.shape)
            unique_values = np.unique(grid)
            value_summary = ",".join(str(x) for x in unique_values[:12])
            parse_status = "parsed numeric grid"
        except Exception as exc:
            grid_shape, value_summary, parse_status = (), "", f"could not parse: {type(exc).__name__}"
        case_id = member.split("/")[0]
        archive_rows.append({"archive": "topology", "member": member, "case_id": case_id, "bytes": len(raw)})
        topology_rows.append({"member": member, "case_id": case_id, "bytes": len(raw), "rows": grid_shape[0] if len(grid_shape) > 0 else np.nan, "columns": grid_shape[1] if len(grid_shape) > 1 else np.nan, "unique_values_first_12": value_summary, "parse_status": parse_status})

with zipfile.ZipFile(archive_specs["netcdf"]) as zf:
    netcdf_members = [name for name in zf.namelist() if name.lower().endswith(".nc") and not name.endswith("/")]
    for member in netcdf_members:
        raw = zf.read(member)
        case_id = member.split("/")[0]
        archive_rows.append({"archive": "netcdf", "member": member, "case_id": case_id, "bytes": len(raw)})
        variable_names, variable_shapes = [], []
        file_format = "unknown"
        try:
            with h5py.File(BytesIO(raw), "r") as h5file:
                def collect_datasets(name, obj):
                    if isinstance(obj, h5py.Dataset):
                        variable_names.append(name)
                        variable_shapes.append(f"{name}:{tuple(obj.shape)}")
                h5file.visititems(collect_datasets)
            file_format = "NetCDF4/HDF5"
        except OSError:
            # The supplied archive mixes HDF5-backed NetCDF4 pedestrian fields and
            # classic NetCDF files for time-series data. Use SciPy for classic files.
            from scipy.io import netcdf_file
            try:
                with netcdf_file(BytesIO(raw), mode="r", mmap=False) as ncfile:
                    for name, variable in ncfile.variables.items():
                        variable_names.append(name)
                        variable_shapes.append(f"{name}:{tuple(variable.shape)}")
                file_format = "classic NetCDF"
            except Exception as exc:
                file_format = f"unreadable: {type(exc).__name__}"
        netcdf_rows.append({"member": member, "case_id": case_id, "bytes": len(raw), "file_format": file_format, "dataset_count": len(variable_names), "datasets": "; ".join(variable_shapes[:30])})

archive_manifest = pd.DataFrame(archive_rows)
profile_crosscheck = pd.DataFrame(profile_crosscheck_rows)
topology_manifest = pd.DataFrame(topology_rows)
netcdf_manifest = pd.DataFrame(netcdf_rows)

archive_manifest.to_csv(DATA_PROCESSED / "raw_archive_manifest.csv", index=False)
profile_crosscheck.to_csv(DATA_PROCESSED / "profile_target_crosscheck.csv", index=False)
topology_manifest.to_csv(DATA_PROCESSED / "topology_archive_manifest.csv", index=False)
netcdf_manifest.to_csv(DATA_PROCESSED / "netcdf_archive_manifest.csv", index=False)

print("Raw archive coverage:")
display(archive_manifest.groupby("archive").agg(files=("member", "count"), cases=("case_id", "nunique"), total_bytes=("bytes", "sum")).reset_index())
print("Sample profile-derived target comparison (diagnostic only; not used to replace the full target column):")
if len(profile_crosscheck):
    display(profile_crosscheck.head(10))
    print("Profile-vs-flow-table MAE (m/s):", round(profile_crosscheck["absolute_difference_m_s"].mean(), 6))
else:
    print("No readable profile/flow target pairs were found.")
print("Topology sample dimensions:")
display(topology_manifest.head())
print("NetCDF sample field inventory:")
display(netcdf_manifest.head())


## 3. Build geometry and layout features from metadata

In [ ]:
def get_family_id(layout_id):
    """Group layouts by footprint + plan-area-density family."""
    match = re.match(r"^[UV]([AS]\d+)", layout_id)
    if not match:
        raise ValueError(f"Unrecognised layout identifier: {layout_id}")
    return match.group(1)


def get_height_pattern(layout_id):
    """Use the folder-code convention: uniform, CL, HR or SM."""
    return "uniform" if layout_id.startswith("U") else layout_id.split("-")[1]


def make_height_features(heights):
    heights = np.asarray(heights, dtype=float)
    mean_h = float(np.mean(heights))
    std_h = float(np.std(heights, ddof=0))
    skew_h = 0.0 if std_h == 0 else float(np.mean(((heights - mean_h) / std_h) ** 3))
    return {
        "height_mean_m": mean_h,
        "height_std_m": std_h,
        "height_max_m": float(np.max(heights)),
        "height_min_m": float(np.min(heights)),
        "height_range_m": float(np.max(heights) - np.min(heights)),
        "height_skew": skew_h,
        "frac_taller_than_mean": float(np.mean(heights > mean_h + 1e-9)),
        "building_count": int(len(heights)),
    }

rows = []
for item in ideal_metadata:
    case_id = item["Folder Name"]
    layout_id = case_id.rsplit("_d", 1)[0]
    heights = item.get("Heights", [])
    if not heights:
        raise ValueError(f"No building-height list in metadata for {case_id}")
    area_density = float(item["areaDensity"])
    row = {
        "simulation_id": case_id,
        "layout_id": layout_id,
        "family_id": get_family_id(layout_id),
        "plan_area_density": area_density,
        "plan_area_density_sq": area_density ** 2,
        **make_height_features(heights),
        "wind_direction_deg": float(item["Wind Direction"]),
        "configuration": "idealized-" + get_height_pattern(layout_id),
        "configuration_metadata_label": str(item.get("Config", "unknown")).rsplit("_d", 1)[0],
        "horizontal_configuration": str(item.get("Horizontal Configuration", "unknown")),
        "density_class": str(item.get("Plan Area Density", "unknown")),
    }
    rows.append(row)

geometry = pd.DataFrame(rows)
print("Geometry rows created from raw metadata:", len(geometry))
display(geometry.head())

## 4. Join the raw flow target and validate case matching

In [ ]:
if "NameE" not in flowdata.columns or "U" not in flowdata.columns:
    raise KeyError("Expected columns 'NameE' and 'U' in UrbanTALES-flowdata.csv")

flow_target = flowdata[["NameE", "U"]].rename(columns={"NameE": "simulation_id", "U": "target_U_canopy_mean_m_s"})
if flow_target["simulation_id"].duplicated().any():
    duplicate_ids = flow_target.loc[flow_target["simulation_id"].duplicated(), "simulation_id"].tolist()
    raise ValueError(f"Duplicate simulation IDs in raw flow data: {duplicate_ids[:5]}")

dataset = geometry.merge(flow_target, on="simulation_id", how="left", validate="one_to_one")
missing_target = dataset["target_U_canopy_mean_m_s"].isna()
if missing_target.any():
    raise ValueError("Raw flow data has no U target for: " + ", ".join(dataset.loc[missing_target, "simulation_id"].head(10)))

# Check the catalogue CSV independently; this catches inconsistent IDs in the raw sources.
if "NameE" in metadata_catalogue.columns:
    catalogue_ids = set(metadata_catalogue["NameE"].astype(str))
    absent_from_catalogue = sorted(set(dataset["simulation_id"]) - catalogue_ids)
    if absent_from_catalogue:
        raise ValueError(f"{len(absent_from_catalogue)} idealized IDs are absent from metadata.csv: {absent_from_catalogue[:5]}")

columns = [
    "simulation_id", "layout_id", "family_id", "target_U_canopy_mean_m_s",
    "plan_area_density", "plan_area_density_sq", "height_mean_m", "height_std_m",
    "height_max_m", "height_min_m", "height_range_m", "height_skew",
    "frac_taller_than_mean", "wind_direction_deg", "configuration",
    "configuration_metadata_label", "horizontal_configuration", "density_class", "building_count"
]
dataset = dataset[columns].sort_values(["layout_id", "wind_direction_deg"]).reset_index(drop=True)

assert len(dataset) == 224, f"Expected 224 idealized cases; got {len(dataset)}"
assert dataset["simulation_id"].nunique() == 224
assert dataset["layout_id"].nunique() == 112
assert dataset["family_id"].nunique() == 16
assert not dataset.isna().any().any(), dataset.isna().sum()[dataset.isna().sum() > 0]
assert set(dataset["wind_direction_deg"].unique()) == {0.0, 45.0}

print("All validation checks passed.")
print(f"Cases: {len(dataset)} | layouts: {dataset.layout_id.nunique()} | families: {dataset.family_id.nunique()}")
display(dataset.head(10))

## 5. Save the newly processed dataset and summary

In [ ]:
processed_path = DATA_PROCESSED / "layout_level_dataset.csv"
summary_path = DATA_PROCESSED / "dataset_summary.json"
dictionary_path = DATA_PROCESSED / "data_dictionary.csv"
dataset.to_csv(processed_path, index=False)

# Generate the data dictionary from the columns actually produced above.
# This keeps the documented schema aligned with the executable dataset builder.
dictionary_details = {
    "simulation_id": ("none", "string", "UrbanTALES simulation case identifier (layout + direction)", "metadata_ideal.json Folder Name", "traceability only; never a predictor"),
    "layout_id": ("none", "string", "Base geometry identifier shared by its wind-direction runs", "Folder Name before direction suffix", "grouping/traceability; not a predictor"),
    "family_id": ("none", "string", "Plan family derived from arrangement letter and plan-area-density code", "derived from layout_id", "primary grouping key for validation; never a predictor"),
    "target_U_canopy_mean_m_s": ("m/s", "float", "Scalar wind-speed target supplied in the raw flow-data table", "UrbanTALES-flowdata.csv column U", "target only; not a predictor"),
    "plan_area_density": ("ratio", "float", "Plan-area density of the idealized array", "metadata_ideal.json areaDensity", "numeric predictor"),
    "plan_area_density_sq": ("ratio^2", "float", "Square of plan-area density to represent nonlinear density effects", "derived from areaDensity", "numeric predictor"),
    "height_mean_m": ("m", "float", "Arithmetic mean building height", "metadata_ideal.json Heights", "retained for traceability; constant in this dataset"),
    "height_std_m": ("m", "float", "Population standard deviation of building heights", "derived from metadata_ideal.json Heights", "numeric predictor"),
    "height_max_m": ("m", "float", "Maximum building height", "derived from metadata_ideal.json Heights", "numeric predictor"),
    "height_min_m": ("m", "float", "Minimum building height", "derived from metadata_ideal.json Heights", "numeric predictor"),
    "height_range_m": ("m", "float", "Maximum minus minimum building height", "derived from metadata_ideal.json Heights", "numeric predictor"),
    "height_skew": ("none", "float", "Population skewness of building heights; zero for uniform heights", "derived from metadata_ideal.json Heights", "numeric predictor"),
    "frac_taller_than_mean": ("fraction", "float", "Share of buildings taller than the mean height", "derived from metadata_ideal.json Heights", "numeric predictor"),
    "wind_direction_deg": ("degrees", "float", "Documented inflow direction", "metadata_ideal.json Wind Direction", "numeric predictor"),
    "configuration": ("category", "string", "Height-pattern category derived from the simulation folder code", "derived from layout_id folder code", "one-hot categorical predictor"),
    "configuration_metadata_label": ("category", "string", "Original catalogue configuration label preserved for traceability", "metadata_ideal.json Config", "traceability only; excluded from predictors"),
    "horizontal_configuration": ("category", "string", "Building arrangement such as aligned or staggered", "metadata_ideal.json Horizontal Configuration", "one-hot categorical predictor"),
    "density_class": ("category", "string", "Categorical density regime from the supplied metadata", "metadata_ideal.json Plan Area Density", "one-hot categorical predictor"),
    "building_count": ("count", "integer", "Number of buildings represented in the height list", "count of metadata_ideal.json Heights", "retained for traceability/baseline ablation; not in engineered predictors"),
}
missing_dictionary_columns = [column for column in dataset.columns if column not in dictionary_details]
if missing_dictionary_columns:
    raise ValueError(f"Add dictionary definitions for new dataset columns: {missing_dictionary_columns}")
dictionary_rows = []
for column in dataset.columns:
    units, data_type, meaning, source, model_use = dictionary_details[column]
    dictionary_rows.append({
        "field": column,
        "units": units,
        "type": data_type,
        "meaning": meaning,
        "source": source,
        "model_use": model_use,
    })
data_dictionary = pd.DataFrame(dictionary_rows, columns=["field", "units", "type", "meaning", "source", "model_use"])
data_dictionary.to_csv(dictionary_path, index=False)

# Keep the human-facing documentation dictionary synchronized with the generated one.
docs_dictionary_path = ROOT / "docs" / "data_dictionary.csv"
docs_dictionary_path.parent.mkdir(parents=True, exist_ok=True)
data_dictionary.to_csv(docs_dictionary_path, index=False)

archive_output_files = [
    "raw_archive_manifest.csv",
    "profile_target_crosscheck.csv",
    "topology_archive_manifest.csv",
    "netcdf_archive_manifest.csv",
]
summary = {
    "source_files": ["data/raw/metadata_ideal.json", "data/raw/metadata.csv", "data/raw/UrbanTALES-flowdata.csv", "data/raw/selected_files.zip", "data/raw/selected_files (1).zip", "data/raw/selected_files (2).zip"],
    "processed_outputs": ["data/processed/layout_level_dataset.csv", "data/processed/data_dictionary.csv", "data/processed/dataset_summary.json"] + [f"data/processed/{name}" for name in archive_output_files],
    "target_source": "UrbanTALES-flowdata.csv column U",
    "target_definition": "Scalar U column from UrbanTALES-flowdata.csv for each idealized case; sample vertical profiles are separately integrated and compared as a diagnostic, not substituted for the full-table targets",
    "cases": int(len(dataset)),
    "layouts": int(dataset["layout_id"].nunique()),
    "plan_families": int(dataset["family_id"].nunique()),
    "wind_directions_deg": sorted(float(x) for x in dataset["wind_direction_deg"].unique()),
    "target_summary": {k: float(v) for k, v in dataset["target_U_canopy_mean_m_s"].describe().to_dict().items()},
    "features_for_models": ["plan_area_density", "plan_area_density_sq", "height_std_m", "height_max_m", "height_min_m", "height_range_m", "height_skew", "frac_taller_than_mean", "wind_direction_deg", "configuration", "horizontal_configuration", "density_class"],
    "excluded_features": {
        "height_mean_m": "constant at 16 m across these idealized cases",
        "building_count": "retained for traceability; omitted from engineered predictor set",
        "simulation_id": "identifier, not a predictor",
        "layout_id": "identifier used for grouping, not a predictor",
        "family_id": "grouping variable used for validation, not a predictor",
        "configuration_metadata_label": "kept for traceability; folder-code configuration is used because CL/SM labels are inconsistent in the catalogue",
    },
    "validation": "224 cases, 112 layouts, 16 families; no missing values; one record per simulation_id. The three ZIP archives are read and audited, but contain sample coverage only.",
    "dictionary_generated_from_dataset_columns": True,
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Created:", processed_path)
print("Created:", dictionary_path)
print("Synchronized:", docs_dictionary_path)
print("Created:", summary_path)
print("Created raw archive audit outputs in:", DATA_PROCESSED)
print("Processed dataset shape:", dataset.shape)
print("Dictionary fields:", len(data_dictionary))
print(json.dumps(summary, indent=2))


cell_typemetadatasource